[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/worldmodels/blob/main/gait/skeleton-jepa/gavd2/00-scan-all-gavd-csvs.ipynb)

# Part 0: Scan every GAVD sequence, at scale (iteration 2)

This is the first notebook of the full-dataset Skeleton-JEPA series, iteration 2.
The concept series one folder up walks a single clip through the whole idea so
you can see what a JEPA is. This series does something different: it scales the
exact same pipeline to every walking sequence in the GAVD dataset, so the model
can learn from all of the cheap unlabeled video before we ever touch the scarce
labels.

Iteration 2 keeps the iteration-1 method unchanged and instead makes the
comparison against the prior Random Forest study genuinely controlled. The prior
work is a Random Forest on 82 hand features that scores 0.76 test accuracy on a
curated 5-class set of 68 sequences. To compare fairly, iteration 2 locks its
labelled probe set to those exact 68 sequences (by sequence id), evaluates one
vector per sequence rather than per overlapping window, and reports the honest
per-sequence number beside the 0.76 baseline. The iteration-1 series lives one
folder over in `gavd/` and is kept as a checkpoint.

The whole plan rests on one imbalance. Unlabeled walking video is plentiful: the
GAVD dataset gives us 374 annotated sequences across 11 gait conditions. Clinical
labels are scarce: only 68 of those sequences make up the clean 5-class set the
prior work trained on. So the strategy is to pretrain on all 374 sequences with
no labels, then spend the 68 labels only at the very end on a tiny probe.

Before we can download or process anything, we need a map of what we have. In
this notebook we walk every condition folder, read every sequence CSV, and build
one tidy manifest table with a single row per sequence. That manifest is the
backbone of the whole series: notebook 01 downloads the unique videos it lists,
notebook 02 extracts skeletons for the sequences it names, and notebook 03 turns
the results into a pretraining corpus. We also visualize the dataset at scale so
you can feel how lopsided the labeled slice really is.

## Run this locally or in Google Colab

There are two ways to run every notebook in this series, and each notebook
repeats these instructions so you never have to leave it.

In Google Colab, click the badge at the top. The first code cell installs
whatever is missing, so you just run the cells from top to bottom.

On your laptop we use [uv](https://docs.astral.sh/uv/), a fast Python package
manager. From a terminal in this `gavd2/` folder:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh   # once, if you do not have uv
uv sync                                            # core deps (smoke mode)
uv run python -m ipykernel install --user --name skeleton-jepa-gavd2 --display-name "Python (skeleton-jepa-gavd2)"
uv run jupyter lab 00-scan-all-gavd-csvs.ipynb
```

Every notebook has a `CONFIG` dictionary whose first key is `SMOKE_TEST`. With
`SMOKE_TEST = True`, the notebook builds a tiny synthetic manifest so it runs in
seconds with no data on disk. With `SMOKE_TEST = False` (the value these committed
notebooks ship with, the real run behind the numbers in the docs), it reads your
real GAVD CSV folder. To scan the real data, copy `.env.example` to `.env`, point
`GAVD_DATA_DIR` at your CSV tree, and keep `SMOKE_TEST = False`.

## The six-notebook journey

The figure below shows how the whole series flows, from this scan all the way to
the frozen-probe evaluation against the 76 percent baseline. Each notebook writes
a small cache file that the next one reads.

![The full-dataset pipeline](images/pipeline-overview.svg)

*Scan all CSVs, bulk-download the unique videos, batch-extract skeletons, build the unlabeled corpus, pretrain the JEPA, then probe the 68 labels against the 76 percent Random Forest baseline.*

## Colab setup

In [ ]:
# Colab setup and local .env loading.
# Installs only missing packages, loads .env for local paths via
# load_dotenv(find_dotenv()), and makes the local alexpose checkout importable.
import importlib.util, subprocess, sys

_import_name = {
    "scikit-learn": "sklearn",
    "opencv-python": "cv2",
    "yt-dlp": "yt_dlp",
    "python-dotenv": "dotenv",
}

def _ensure(pkgs):
    """pip install any packages whose import is not already available."""
    missing = [p for p in pkgs if importlib.util.find_spec(_import_name.get(p, p)) is None]
    if missing:
        print("Installing:", " ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    return missing

# Core packages, always needed (smoke mode uses only these).
_ensure(["numpy", "pandas", "matplotlib", "python-dotenv", "tqdm"])

# This notebook only reads CSV files, so it needs no REAL-path video packages.

# Credentials, keys, and local paths are loaded with load_dotenv(find_dotenv()).
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())          # searches upward for .env; no-op if absent
print("Loaded environment via load_dotenv(find_dotenv()).")

ALEXPOSE_REPO = os.getenv("ALEXPOSE_REPO")
GAVD_DATA_DIR = os.getenv("GAVD_DATA_DIR")
YOUTUBE_CACHE_DIR = os.getenv("YOUTUBE_CACHE_DIR")
GAVD_CACHE_DIR = os.getenv("GAVD_CACHE_DIR")
DEMO_VIDEO_ID = os.getenv("DEMO_VIDEO_ID", "B5hrxKe2nP8")
# iteration-2 keys: the curated exp5 labelled tree and the exp5 82-feature pickle,
# used by the exact-68 id resolver below to lock the labelled probe set.
EXP4_DATA_DIR = os.getenv("EXP4_DATA_DIR")
EXP5_FEATURES_PKL = os.getenv("EXP5_FEATURES_PKL")
if ALEXPOSE_REPO and os.path.isdir(ALEXPOSE_REPO) and ALEXPOSE_REPO not in sys.path:
    sys.path.insert(0, ALEXPOSE_REPO)
print("Setup complete.")

## Configuration

Everything you might want to change lives in one `CONFIG` dictionary, and
`SMOKE_TEST` is the first key. The cache directory is where this series stores
its own intermediate files. We keep it separate from the alexpose data so you
can delete it freely without touching the source CSVs or downloaded videos.

In [ ]:
from pathlib import Path
import hashlib

CONFIG = {
    "SMOKE_TEST": False,          # True -> tiny synthetic manifest. False -> read the real CSV tree.
    # Where the per-condition CSV folders live (from .env, with a sensible default).
    "GAVD_DIR": Path(GAVD_DATA_DIR) if GAVD_DATA_DIR else Path.home() / "dev" / "alexpose" / "data" / "gavd",
    # Where this series caches its own artifacts (manifest, skeletons, corpus, encoder).
    # Blank GAVD_CACHE_DIR -> gavd2/cache, which keeps iteration 2 independent of gavd/.
    "CACHE_DIR": Path(GAVD_CACHE_DIR) if GAVD_CACHE_DIR else Path.cwd() / "cache",
    # The curated exp5 labelled tree and the exp5 82-feature pickle for the exact-68 lock.
    "EXP4_DATA_DIR": Path(EXP4_DATA_DIR) if EXP4_DATA_DIR else (Path(ALEXPOSE_REPO) / "experiments" / "exp4" / "data" if ALEXPOSE_REPO else None),
    "EXP5_FEATURES_PKL": Path(EXP5_FEATURES_PKL) if EXP5_FEATURES_PKL else (Path(ALEXPOSE_REPO) / "experiments" / "exp5" / "features" / "all82_features.pkl" if ALEXPOSE_REPO else None),
    # The 5-class clinically labeled subset the prior work used, with target counts.
    # Keys are CANONICAL spellings (cerebralpalsy, no space) to match exp5's folders/pickle.
    "LABELED_CLASSES": {"normal": 12, "parkinsons": 9, "stroke": 12, "cerebralpalsy": 15, "myopathic": 20},
    "BASELINE_ACC": 0.76,        # prior Random Forest test accuracy to beat
    # When True (and REAL), fall back to the iteration-1 first-N-per-class labelling,
    # writing to a SEPARATE cache namespace so it never contaminates the locked run.
    "EXPLORATORY_FIRST_N": False,
}
# Cache namespace: "" for the locked (exp5-exact) run, "_firstN" for the exploratory run.
# Every artifact filename in the series carries this suffix so the two never mix.
CONFIG["CACHE_NS"] = "_firstN" if CONFIG["EXPLORATORY_FIRST_N"] else ""
CONFIG["CACHE_DIR"].mkdir(parents=True, exist_ok=True)

# Shared helpers threaded through every notebook in the series. They are cheap and
# self-contained so each notebook stays runnable on its own.

# Canonicalize condition spellings so the cerebral-palsy class never silently drops.
# The full GAVD tree names the folder "cerebral palsy" (with a space); exp5's curated
# tree and its feature pickle use "cerebralpalsy" (no space). We canonicalize the LABEL
# everywhere; a separate map (below) recovers the on-disk FOLDER spelling for file reads.
CANONICAL_COND = {"cerebral palsy": "cerebralpalsy"}
def canon_cond(c):
    c = str(c).strip().lower()
    return CANONICAL_COND.get(c, c.replace(" ", ""))

# Canonical label -> on-disk GAVD folder name (inverse of canon for the space case).
# Used by notebook 02 to build the extraction path; canonicalizing the label but not
# the path would look up a folder that does not exist and drop the class to zero.
COND_TO_FOLDER = {"cerebralpalsy": "cerebral palsy"}

# Deterministic fingerprint of the locked 68 ids, stamped onto every cache artifact so a
# downstream notebook can detect (and warn about) a stale-artifact mix.
def canonical_id_hash(ids):
    return hashlib.sha1("\n".join(sorted(map(str, ids))).encode()).hexdigest()[:12]

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k:18s} = {v}")
print(f"\n68 labeled sequences across 5 classes: {sum(CONFIG['LABELED_CLASSES'].values())} total")
print(f"Cache namespace: '{CONFIG['CACHE_NS']}'  (locked run writes un-suffixed artifacts)")

## What one GAVD CSV looks like

Each sequence is one CSV file, and each row in it is one annotated frame of one
person walking in one YouTube video. The ten columns tell us which sequence and
frame we are on, where the person is in the frame (the bounding box), the video
size, the YouTube id and url, and the gait condition label. The condition also
happens to be the name of the folder the CSV sits in.

To keep this notebook runnable with no data on disk, the helper below builds a
tiny synthetic CSV in memory in smoke mode, and reads a real one in real mode.
Either way you see the same ten columns, so the rest of the notebook does not
care which path produced the table.

In [ ]:
import pandas as pd
import ast

GAVD_COLUMNS = ["seq", "frame_num", "cam_view", "gait_event", "dataset",
                "gait_pat", "bbox", "vid_info", "id", "url"]

def make_synthetic_sequence_df(seq_id, video_id, condition, n_frames=40, start_frame=1):
    """Build one synthetic sequence DataFrame with the exact GAVD 10 columns."""
    rows = []
    for k in range(n_frames):
        f = start_frame + k
        # A slowly drifting bounding box, stored as a python-dict string like real GAVD.
        bbox = {"top": 120.0 + 0.1 * k, "left": 150.0 + 0.5 * k, "height": 495.0, "width": 230.0}
        vinfo = {"height": 720, "width": 1280, "mime_type": "video/mp4"}
        rows.append({
            "seq": seq_id, "frame_num": f, "cam_view": "right side", "gait_event": "",
            "dataset": "Abnormal Gait", "gait_pat": condition,
            "bbox": str(bbox), "vid_info": str(vinfo), "id": video_id,
            "url": f"https://www.youtube.com/watch?v={video_id}",
        })
    return pd.DataFrame(rows, columns=GAVD_COLUMNS)

if CONFIG["SMOKE_TEST"]:
    demo_df = make_synthetic_sequence_df("clsynth00demo0000000000000", DEMO_VIDEO_ID, "parkinsons")
    print(f"SMOKE mode: built a synthetic sequence DataFrame (demo video id {DEMO_VIDEO_ID}).")
else:
    # REAL mode: read one real CSV so you can see the true schema.
    example_csv = next(iter(sorted((CONFIG["GAVD_DIR"] / "parkinsons").glob("*.csv"))), None)
    if example_csv is None:
        raise FileNotFoundError(f"No CSVs under {CONFIG['GAVD_DIR']}. Check GAVD_DATA_DIR in your .env.")
    demo_df = pd.read_csv(example_csv)
    print(f"REAL mode: read {example_csv.name}")

print(f"\nShape: {demo_df.shape}  (rows = frames, columns = the 10 GAVD fields)")
print("Columns:", list(demo_df.columns))
demo_df.head(3)

## Reading the bounding box and the video id

Two columns hold small dictionaries stored as text. The `bbox` column is where
the walking person sits inside the frame, in pixels, as `top`, `left`, `height`,
`width`. The `vid_info` column holds the source video width and height. We parse
both with `ast.literal_eval`, which safely turns a python-dict string back into a
real dictionary. Notebook 02 uses the box to crop to the person before running
pose estimation, and it uses the video size to keep every joint in whole-frame
normalized coordinates.

In [ ]:
row0 = demo_df.iloc[0]
bbox = ast.literal_eval(row0["bbox"])
vid_info = ast.literal_eval(row0["vid_info"])

print("First frame of this sequence:")
print(f"  video id      : {row0['id']}")
print(f"  url           : {row0['url']}")
print(f"  condition     : {row0['gait_pat']}")
print(f"  frame_num     : {row0['frame_num']}  (1-based within the source video)")
print(f"  bounding box  : top={bbox['top']:.0f}, left={bbox['left']:.0f}, "
      f"height={bbox['height']:.0f}, width={bbox['width']:.0f}  (pixels)")
print(f"  source video  : {vid_info['width']} x {vid_info['height']}")

## The body behind the rows

Every row we just scanned is one frame of a person walking. To keep that concrete
while we work with tables, we animate a walking skeleton inline, the same kind of
motion the later notebooks turn into data. In this scan notebook we always show a
small synthetic walker so nothing has to download; the extraction notebook plays
the real skeletons it pulls from the videos.

In [ ]:
# Inline animation helper: animate_skeleton (self-contained, works in Jupyter and Colab).
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

EDGES = [
    (0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(0,9),(0,10),(9,10),
    (11,12),(11,23),(12,24),(23,24),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]

def synthesize_walking_skeleton(T=16, seed=0):
    """A plausible synthetic (T, 33, 3) walking skeleton for the preview animation."""
    rng = np.random.RandomState(seed)
    base = np.zeros((33, 3), dtype=np.float32)
    # Full (x, y) layout for all 33 landmarks, laid out as a person seen head-on.
    # y grows downward (head near 0.16, feet near 0.97). x has the midline at 0.50,
    # with the left side (odd joint indices) left of it and the right side right of it.
    # Shoulders are wider than the hips, the arms hang OUTSIDE the hips down to about
    # hip height, and the head sits just above the shoulders, so the figure reads as a
    # real walking body instead of collapsing onto one vertical line.
    xs = {
        0:0.500,                                            # nose
        1:0.485, 2:0.475, 3:0.465, 4:0.515, 5:0.525, 6:0.535,  # eyes (left then right)
        7:0.455, 8:0.545,                                   # ears
        9:0.485, 10:0.515,                                  # mouth
        11:0.415, 12:0.585,                                 # shoulders (wide)
        13:0.395, 14:0.605,                                 # elbows (arms hang outside)
        15:0.405, 16:0.595,                                 # wrists
        17:0.395, 18:0.605, 19:0.405, 20:0.595, 21:0.420, 22:0.580,  # hands track their wrist
        23:0.455, 24:0.545,                                 # hips (narrower than shoulders)
        25:0.450, 26:0.550,                                 # knees
        27:0.448, 28:0.552,                                 # ankles
        29:0.448, 30:0.552, 31:0.455, 32:0.545,             # heels, foot tips
    }
    ys = {
        0:0.16,                                             # nose
        1:0.145, 2:0.145, 3:0.145, 4:0.145, 5:0.145, 6:0.145,  # eyes
        7:0.155, 8:0.155,                                   # ears
        9:0.185, 10:0.185,                                  # mouth (short neck to shoulders)
        11:0.24, 12:0.24,                                   # shoulders
        13:0.38, 14:0.38,                                   # elbows
        15:0.51, 16:0.51,                                   # wrists (about hip height)
        17:0.545, 18:0.545, 19:0.545, 20:0.545, 21:0.535, 22:0.535,  # hands (just past wrists)
        23:0.50, 24:0.50,                                   # hips
        25:0.71, 26:0.71,                                   # knees
        27:0.92, 28:0.92,                                   # ankles
        29:0.94, 30:0.94, 31:0.965, 32:0.965,               # heels, foot tips
    }
    for j in range(33):
        base[j, 0] = xs[j]
        base[j, 1] = ys[j]
    seq = np.repeat(base[None], T, axis=0)
    t = np.linspace(0, 2*np.pi, T, endpoint=False)
    swing = 0.07 * np.sin(t)
    for k, amp in [(25,1.0),(27,1.3),(31,1.4),(13,-0.8),(15,-1.0)]:
        seq[:, k, 0] += swing * amp
    for k, amp in [(26,-1.0),(28,-1.3),(32,-1.4),(14,0.8),(16,1.0)]:
        seq[:, k, 0] += swing * amp
    seq += rng.randn(T, 33, 3).astype(np.float32) * 0.004
    return seq

def animate_skeleton(seq, edges, title="Walking skeleton", fps=8):
    """Animate a (T, 33, C) skeleton inline (uses x=seq[...,0], y=seq[...,1])."""
    T = seq.shape[0]; x_all = seq[:, :, 0]; y_all = seq[:, :, 1]
    groups = [list(range(11)), [11,13,15,17,19,21], [12,14,16,18,20,22],
              [11,12,23,24], [23,25,27,29,31], [24,26,28,30,32]]
    colors = ["#8b5cf6","#3b82f6","#ef4444","#22c55e","#f59e0b","#ec4899"]
    fig, ax = plt.subplots(figsize=(6, 7))
    x_min, x_max = x_all.min(), x_all.max(); y_min, y_max = y_all.min(), y_all.max()
    margin = max(x_max - x_min, y_max - y_min) * 0.1 + 1e-3
    def draw_frame(t):
        ax.clear(); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
        ax.set_xlim(x_min - margin, x_max + margin); ax.set_ylim(y_max + margin, y_min - margin)
        ax.set_title(f"{title} (frame {t}/{T})")
        x = x_all[t]; y = y_all[t]
        for g_idx, grp in enumerate(groups):
            for (i, j) in [(i, j) for (i, j) in edges if i in grp and j in grp]:
                ax.plot([x[i], x[j]], [y[i], y[j]], color=colors[g_idx], linewidth=2, alpha=0.7)
            ax.scatter([x[i] for i in grp], [y[i] for i in grp], c=colors[g_idx],
                       s=40, zorder=3, edgecolors='white', linewidths=0.5)
    anim = FuncAnimation(fig, draw_frame, frames=T, interval=1000/fps, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

print("animate_skeleton helper defined.")

In [ ]:
demo_skeleton = synthesize_walking_skeleton(T=16, seed=0)
print("A synthetic walking skeleton, the motion behind every row of the manifest.")
display(animate_skeleton(demo_skeleton, EDGES, title="A walking sequence"))

## Walk every condition folder, build the manifest

Now the real work of this notebook. We iterate over every condition folder, read
every sequence CSV, and reduce each one to a single manifest row: which condition
it belongs to, its sequence id, the YouTube video id and url, the first and last
frame numbers, how many frames it spans, and whether it carries bounding boxes.
We do not load the frames into memory, we only summarize, so this stays fast even
across hundreds of CSVs.

In smoke mode we fabricate a small manifest that mimics the real class balance in
miniature, so the plots and the labeled-subset logic below have something to
chew on. In real mode we use the alexpose `GAVDDataLoader` when its `ambient`
package is importable, and fall back to a plain `pandas.read_csv` otherwise, so
the scan works whether or not you have the private repo checked out.

In [ ]:
import string

def summarize_sequence(df):
    """Reduce a sequence DataFrame to one manifest row (a dict).
    The condition is CANONICALIZED here (cerebral palsy -> cerebralpalsy) so the
    5-class label spelling matches exp5 everywhere downstream."""
    frames = pd.to_numeric(df["frame_num"], errors="coerce").dropna().astype(int)
    return {
        "seq": str(df["seq"].iloc[0]),
        "condition": canon_cond(df["gait_pat"].iloc[0]),
        "video_id": str(df["id"].iloc[0]),
        "url": str(df["url"].iloc[0]),
        "start_frame": int(frames.min()) if len(frames) else 0,
        "end_frame": int(frames.max()) if len(frames) else 0,
        "num_frames": int(len(df)),
        "has_bbox": bool(df["bbox"].notna().all()),
    }

def build_real_manifest(gavd_dir):
    """Scan gavd_dir/<condition>/*.csv into a manifest DataFrame."""
    # Prefer the alexpose loader; fall back to pandas if ambient is unavailable.
    try:
        from ambient.gavd import GAVDDataLoader
        loader = GAVDDataLoader()
        read_one = lambda p: loader.load_gavd_data(str(p))
        print("Using ambient.gavd.GAVDDataLoader to read CSVs.")
    except Exception:
        read_one = lambda p: pd.read_csv(p)
        print("ambient not available; reading CSVs with pandas.read_csv.")

    condition_dirs = sorted(p for p in gavd_dir.iterdir()
                            if p.is_dir() and p.name[0] in string.ascii_letters)
    rows = []
    for cond in condition_dirs:
        csvs = sorted(cond.glob("*.csv"))
        for csv_path in csvs:
            try:
                df = read_one(csv_path)
                rows.append(summarize_sequence(df))
            except Exception as e:
                print(f"  skipped {csv_path.name}: {e}")
        print(f"  {cond.name:16s}: {len(csvs)} sequences")
    return pd.DataFrame(rows)

def build_synthetic_manifest():
    """A miniature manifest that mirrors the real class balance for smoke mode.
    Conditions use canonical spellings so smoke and real agree on the 5 labels."""
    # Real counts scaled down but keeping the shape (abnormal dominates, labeled classes small).
    counts = {"abnormal": 12, "style": 6, "myopathic": 5, "exercise": 3, "cerebralpalsy": 3,
              "normal": 3, "stroke": 3, "parkinsons": 2, "prosthetic": 1, "antalgic": 1, "inebriated": 1}
    rows = []
    vid = 0
    for cond, n in counts.items():
        for i in range(n):
            # Let some sequences share a video id, as in real GAVD.
            vid_id = f"vid{vid // 2:05d}"
            vid += 1
            rows.append({
                "seq": f"cl{cond[:3]}{i:04d}synthetic00000000",
                "condition": cond, "video_id": vid_id,
                "url": f"https://www.youtube.com/watch?v={vid_id}",
                "start_frame": 100 + 10 * i, "end_frame": 100 + 10 * i + 39,
                "num_frames": 40, "has_bbox": True,
            })
    return pd.DataFrame(rows)

if CONFIG["SMOKE_TEST"]:
    manifest = build_synthetic_manifest()
    print("SMOKE mode: built a synthetic manifest that mimics the real class balance.")
else:
    manifest = build_real_manifest(CONFIG["GAVD_DIR"])

print(f"\nManifest: {len(manifest)} sequences, {manifest['video_id'].nunique()} unique videos")
manifest.head()

## Lock the labelled set to the exact exp5 68

Iteration 1 marked the labelled subset by taking the first N sequences of each
class. That is close but not identical to the 68 sequences the prior Random
Forest actually trained on, so the comparison was not truly controlled. Iteration
2 fixes this: it resolves the exact 68 sequence ids the exp5 study used and marks
only those as labelled. Everything else, all the other sequences, still feeds
unlabelled pretraining.

The resolver tries three sources in order so it runs anywhere. First it unpickles
the exp5 82-feature file and reads each sequence id and label (this also gives us
the exact list order exp5 used, which we need to reproduce its train/test split).
That unpickle needs the private ambient package, so in this notebook's own
environment it fails and we fall through. Second it globs the five curated exp5
CSV folders, whose file names are the sequence ids. Third, for a bare Colab with
no exp5 files at all, it uses a checked-in constant of the 68 ids. A real locked
run stops with a clear error if all three fail, rather than silently falling back
to the old first-N behaviour.

In [ ]:
# Resolve the EXACT 68 exp5 labelled sequence ids (the controlled-comparison lock).
#
# Three tiers, tried in order, so this runs whether you have the exp5 pickle, just
# the curated CSV folders, or neither (Colab). The pickle also carries the ORDER
# exp5 used, which we need to reproduce its exact seed-42 train/test split.
import numpy as np

# Tier 3 constant: the 68 ids -> canonical class, checked in so a bare Colab works.
# Generated once from the exp5 pickle; regenerate if the curated 68 ever change.
CANONICAL_68 = {
    # normal (12)
    "cljo340cm002a3n6low42ugvh": "normal",
    "cljo32xnz00223n6lvxzyif3y": "normal",
    "cljo33m8400263n6l4xxsl6ku": "normal",
    "cljo2y1f7001e3n6lt1wgacw6": "normal",
    "cljo3b2dy002l3n6l270vpzp8": "normal",
    "cljo2yqzp001i3n6lg75p7wtq": "normal",
    "cljo39ok9002h3n6ldr0w5sey": "normal",
    "cljo30lnz001q3n6lopfty7q5": "normal",
    "cljo2wwu7001a3n6ljmqm39l6": "normal",
    "cljo2zn41001m3n6lhbvww48i": "normal",
    "cljo32ik2001y3n6lmmnu0sgo": "normal",
    "cljo32213001u3n6lel97up5f": "normal",
    # parkinsons (9)
    "cljnz7jnj000w3n6lku3pfvtm": "parkinsons",
    "cljnz6vg1000s3n6lxxknbe72": "parkinsons",
    "cljnz3l34000c3n6ldapq560j": "parkinsons",
    "cljnz5sb1000o3n6lntosswwz": "parkinsons",
    "cljan9b4p00043n6ligceanyp": "parkinsons",
    "cljanb45y00083n6lmh1qhydd": "parkinsons",
    "cljnyzwbo00043n6lugyldlhu": "parkinsons",
    "cljnz4y5a000k3n6lkv4b1rjn": "parkinsons",
    "cljnz4e8u000g3n6l1luikppo": "parkinsons",
    # stroke (12)
    "cljvvsucg00043n6l4evgn7q4": "stroke",
    "cljr5jk0h000n3n6la34mkdfz": "stroke",
    "cljo8eumx00683n6le2s7myd4": "stroke",
    "cljr5fc5d000b3n6lkvc71zyl": "stroke",
    "cljr5iki0000j3n6lwi8z5nh6": "stroke",
    "cljo8cyv500603n6lyl148tmg": "stroke",
    "cljo8g74m006g3n6l6kuxy9cf": "stroke",
    "cljo8hcfv006k3n6lgrx0fcpx": "stroke",
    "cljo8fdke006c3n6lr8bzjjgi": "stroke",
    "cljo8c0sw005w3n6l9ulr2eg2": "stroke",
    "cljo8e32t00643n6l37ncjeic": "stroke",
    "cljr5hwxc000f3n6lof5w9tyt": "stroke",
    # cerebralpalsy (15)
    "cljar9bqo00c43n6l2u5zmlru": "cerebralpalsy",
    "cljarbn1y00cg3n6l1u4i0d5l": "cerebralpalsy",
    "cljardvzg00cs3n6loetskba6": "cerebralpalsy",
    "cljas71p600fv3n6lk1rzl7y5": "cerebralpalsy",
    "cljas134500f73n6lkfbjfayp": "cerebralpalsy",
    "cljarcfa700ck3n6lfww83ig1": "cerebralpalsy",
    "cljar9t8o00c83n6ltculhoct": "cerebralpalsy",
    "cljas1yfs00fb3n6lna38ui6i": "cerebralpalsy",
    "cljarar9t00cc3n6lqhi9udoc": "cerebralpalsy",
    "cljas5esv00fn3n6lewd5xqdl": "cerebralpalsy",
    "cljas4dqj00fj3n6ldw6wiwpy": "cerebralpalsy",
    "cljar878f00c03n6ly2v2ay88": "cerebralpalsy",
    "cljas04fw00f33n6lm5cvx9g6": "cerebralpalsy",
    "cljas2sou00ff3n6lasppj8h2": "cerebralpalsy",
    "cljarcy3g00co3n6lzsn1x034": "cerebralpalsy",
    # myopathic (20)
    "cljarhldg00d13n6l7utw0lqn": "myopathic",
    "cljo83yl800513n6lfglf5jn8": "myopathic",
    "cljawdoej000d3n6ll5ysj34f": "myopathic",
    "cljarmcm700dh3n6lxw24hxgx": "myopathic",
    "cljaxabfg003a3n6l95mrlcry": "myopathic",
    "cljarj6rf00d53n6ljtivf4q7": "myopathic",
    "cljarlbch00dd3n6l9jaubrxq": "myopathic",
    "cljarp08600dt3n6lxa7sysiv": "myopathic",
    "cljarpts600dx3n6l98zqf6yn": "myopathic",
    "cljaroguw00dp3n6lhfddiadu": "myopathic",
    "cljargosy00cx3n6lm6atozrz": "myopathic",
    "cljarn9oy00dl3n6l8pg9exfg": "myopathic",
    "cljarqtuz00e13n6lrnox6mfs": "myopathic",
    "cljaxbq22003i3n6lmzka93uq": "myopathic",
    "cljarkax200d93n6lukx52g1t": "myopathic",
    "cljax9d2p00363n6lx043s1m7": "myopathic",
    "cljaxb5y2003e3n6lj1j9qvav": "myopathic",
    "cljo84jju00553n6lmgm4dqtb": "myopathic",
    "cljawd01m00093n6l4kx9020l": "myopathic",
    "cljawb5nf00043n6l38chxsod": "myopathic",
}

# exp5's native all_features list order (positions 0..67). exp5's get_train_test does
# np.random.seed(42); permutation(len); first 70% train / last 30% test, permuting these
# POSITIONS. Reproducing the exact 47/21 split therefore needs this exact ordering, not a
# sorted id set. Tier 1 recovers it live from the pickle; this checked-in copy is the
# Colab fallback and must match the pickle order.
EXP5_ORDER = [
    "cljvvsucg00043n6l4evgn7q4", "cljr5jk0h000n3n6la34mkdfz", "cljo8eumx00683n6le2s7myd4",
    "cljr5fc5d000b3n6lkvc71zyl", "cljr5iki0000j3n6lwi8z5nh6", "cljo8cyv500603n6lyl148tmg",
    "cljo8g74m006g3n6l6kuxy9cf", "cljo8hcfv006k3n6lgrx0fcpx", "cljo8fdke006c3n6lr8bzjjgi",
    "cljo8c0sw005w3n6l9ulr2eg2", "cljo8e32t00643n6l37ncjeic", "cljr5hwxc000f3n6lof5w9tyt",
    "cljarhldg00d13n6l7utw0lqn", "cljo83yl800513n6lfglf5jn8", "cljawdoej000d3n6ll5ysj34f",
    "cljarmcm700dh3n6lxw24hxgx", "cljaxabfg003a3n6l95mrlcry", "cljarj6rf00d53n6ljtivf4q7",
    "cljarlbch00dd3n6l9jaubrxq", "cljarp08600dt3n6lxa7sysiv", "cljarpts600dx3n6l98zqf6yn",
    "cljaroguw00dp3n6lhfddiadu", "cljargosy00cx3n6lm6atozrz", "cljarn9oy00dl3n6l8pg9exfg",
    "cljarqtuz00e13n6lrnox6mfs", "cljaxbq22003i3n6lmzka93uq", "cljarkax200d93n6lukx52g1t",
    "cljax9d2p00363n6lx043s1m7", "cljaxb5y2003e3n6lj1j9qvav", "cljo84jju00553n6lmgm4dqtb",
    "cljawd01m00093n6l4kx9020l", "cljawb5nf00043n6l38chxsod", "cljnz7jnj000w3n6lku3pfvtm",
    "cljnz6vg1000s3n6lxxknbe72", "cljnz3l34000c3n6ldapq560j", "cljnz5sb1000o3n6lntosswwz",
    "cljan9b4p00043n6ligceanyp", "cljanb45y00083n6lmh1qhydd", "cljnyzwbo00043n6lugyldlhu",
    "cljnz4y5a000k3n6lkv4b1rjn", "cljnz4e8u000g3n6l1luikppo", "cljo340cm002a3n6low42ugvh",
    "cljo32xnz00223n6lvxzyif3y", "cljo33m8400263n6l4xxsl6ku", "cljo2y1f7001e3n6lt1wgacw6",
    "cljo3b2dy002l3n6l270vpzp8", "cljo2yqzp001i3n6lg75p7wtq", "cljo39ok9002h3n6ldr0w5sey",
    "cljo30lnz001q3n6lopfty7q5", "cljo2wwu7001a3n6ljmqm39l6", "cljo2zn41001m3n6lhbvww48i",
    "cljo32ik2001y3n6lmmnu0sgo", "cljo32213001u3n6lel97up5f", "cljar9bqo00c43n6l2u5zmlru",
    "cljarbn1y00cg3n6l1u4i0d5l", "cljardvzg00cs3n6loetskba6", "cljas71p600fv3n6lk1rzl7y5",
    "cljas134500f73n6lkfbjfayp", "cljarcfa700ck3n6lfww83ig1", "cljar9t8o00c83n6ltculhoct",
    "cljas1yfs00fb3n6lna38ui6i", "cljarar9t00cc3n6lqhi9udoc", "cljas5esv00fn3n6lewd5xqdl",
    "cljas4dqj00fj3n6ldw6wiwpy", "cljar878f00c03n6ly2v2ay88", "cljas04fw00f33n6lm5cvx9g6",
    "cljas2sou00ff3n6lasppj8h2", "cljarcy3g00co3n6lzsn1x034",
]

CURATED_CLASSES = ["normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"]

def resolve_exp5_labeled(cfg):
    """Return (labeled_id_to_class dict, exp5_order list, source string).
    labeled_id_to_class has exactly 68 entries; exp5_order preserves exp5's list order."""
    pkl = cfg.get("EXP5_FEATURES_PKL")
    # Tier 1: unpickle exp5 all82_features.pkl. The pickle is
    # {"all_features": [GaitFeatureVector x68], "condition_counts": {...}, "metadata": {...}};
    # ids live in f.sample_id and labels in f.condition_label. Unpickling needs the ambient
    # package on sys.path, so in the gavd2 venv this raises and we fall through (expected).
    try:
        import pickle
        with open(pkl, "rb") as fh:
            obj = pickle.load(fh)
        feats = obj["all_features"]
        order = [str(f.sample_id) for f in feats]
        id2c = {str(f.sample_id): canon_cond(f.condition_label) for f in feats}
        if len(id2c) == 68:
            return id2c, order, "tier1:exp5-pkl"
    except Exception as e:
        print(f"  tier1 (exp5 pickle) unavailable: {type(e).__name__}: {str(e)[:80]}")
    # Tier 2: glob ONLY the five curated class folders under EXP4_DATA_DIR; the CSV stems
    # are the sequence ids. Folder names use the canonical spelling (cerebralpalsy).
    try:
        e4 = cfg.get("EXP4_DATA_DIR")
        if e4 and Path(e4).is_dir():
            id2c = {}
            for c in CURATED_CLASSES:
                folder = Path(e4) / c
                for p in sorted(folder.glob("*.csv")):
                    id2c[p.stem] = c
            if len(id2c) == 68:
                # exp4/data has no canonical ORDER, so reuse the checked-in EXP5_ORDER
                # (which came from the pickle) for the exact-split reconstruction.
                return id2c, list(EXP5_ORDER), "tier2:exp4-glob"
            else:
                print(f"  tier2 (exp4 glob) found {len(id2c)} ids, expected 68; falling through.")
    except Exception as e:
        print(f"  tier2 (exp4 glob) unavailable: {type(e).__name__}: {str(e)[:80]}")
    # Tier 3: checked-in constant (Colab / no exp5 artifacts on disk).
    if len(CANONICAL_68) == 68:
        return dict(CANONICAL_68), list(EXP5_ORDER), "tier3:canonical-constant"
    return None, None, "none"

def exp5_exact_split(order, train_portion=0.7):
    """Reproduce exp5's exact seed-42 split over exp5's list ORDER: first 70% train,
    last 30% test of a seed-42 permutation of positions. Returns (train_ids, test_ids)."""
    np.random.seed(42)
    perm = np.random.permutation(len(order))
    k = int(train_portion * len(order))
    train_ids = [order[i] for i in perm[:k]]
    test_ids = [order[i] for i in perm[k:]]
    return train_ids, test_ids

# In SMOKE mode we do not lock to exp5 (the synthetic manifest has no exp5 ids); the
# labelled marking below stays synthetic. In REAL mode we resolve the exact 68.
if CONFIG["SMOKE_TEST"] or CONFIG["EXPLORATORY_FIRST_N"]:
    LABELED_ID2CLASS, EXP5_LIST_ORDER, LABEL_SRC = None, None, ("smoke" if CONFIG["SMOKE_TEST"] else "exploratory-firstN")
else:
    LABELED_ID2CLASS, EXP5_LIST_ORDER, LABEL_SRC = resolve_exp5_labeled(CONFIG)
    assert LABELED_ID2CLASS is not None, (
        "REAL locked run FAIL-STOP: all three exp5-id tiers failed to resolve 68 ids. "
        "Check EXP4_DATA_DIR / EXP5_FEATURES_PKL in your .env, or set EXPLORATORY_FIRST_N=True "
        "to run the exploratory first-N labelling in a separate cache namespace.")
print(f"Labelled-id source: {LABEL_SRC}; "
      f"N={0 if LABELED_ID2CLASS is None else len(LABELED_ID2CLASS)}")

## The dataset at scale, and the labeled slice inside it

The plots below make the central imbalance visible. On the left, the number of
sequences per condition: a few conditions like abnormal and style dominate, and
the clinically clean classes are small. On the right, the same conditions split
into the 68 labeled sequences the prior work used versus everything else. That
gray mountain of unlabeled sequences is exactly what a JEPA can learn from and a
label-hungry classifier cannot.

In [ ]:
import matplotlib.pyplot as plt

# Mark the labelled subset. In the locked REAL run this is the EXACT exp5 68 (by
# sequence id) resolved above, so the comparison is controlled. In SMOKE or the
# exploratory first-N run there is no exp5 lock, so we keep the iteration-1
# first-N-per-class marking. Everything not marked feeds unlabelled pretraining.
labeled_classes = CONFIG["LABELED_CLASSES"]
manifest = manifest.sort_values(["condition", "seq"]).reset_index(drop=True)
manifest["is_labeled"] = False

if LABELED_ID2CLASS is not None:
    # Locked exp5 path: mark exactly the resolved 68 sequence ids.
    labeled_ids = set(LABELED_ID2CLASS)
    manifest.loc[manifest["seq"].astype(str).isin(labeled_ids), "is_labeled"] = True
    matched = int(manifest["is_labeled"].sum())
    # Any of the 68 that are not in the scanned tree would break the lock; report them.
    missing = sorted(labeled_ids - set(manifest["seq"].astype(str)))
    if missing:
        print(f"WARNING: {len(missing)} of the 68 exp5 ids were not found in the scanned tree:")
        for m in missing[:10]:
            print(f"    {m}")
    # The exp5-lock assertions run in REAL mode only, so SMOKE stays green.
    if not CONFIG["SMOKE_TEST"]:
        assert matched == 68, f"exp5 lock: expected 68 labelled sequences, matched {matched}"
        per_class = manifest[manifest["is_labeled"]]["condition"].value_counts().to_dict()
        for cls, n in CONFIG["LABELED_CLASSES"].items():
            assert per_class.get(cls, 0) == n, (
                f"exp5 lock: class {cls} has {per_class.get(cls, 0)} labelled, expected {n} "
                f"(a spelling drift or a missing sequence)")
        print(f"exp5 lock verified: {matched} labelled sequences, per class {per_class}.")
else:
    # SMOKE / exploratory: first-N-per-class, exactly as iteration 1.
    for cond, n in labeled_classes.items():
        idx = manifest.index[manifest["condition"] == cond][:n]
        manifest.loc[idx, "is_labeled"] = True
    print("SMOKE / exploratory: labelled by first-N-per-class (not the exp5 lock).")

# Every labelled row also carries the canonical exp5 class label for auditing.
manifest["in_exp5_curated"] = manifest["is_labeled"]

counts = manifest["condition"].value_counts().sort_values(ascending=True)
labeled_counts = manifest[manifest["is_labeled"]]["condition"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(counts.index, counts.values, color="#3b82f6")
axes[0].set_title(f"Sequences per condition (total {len(manifest)})")
axes[0].set_xlabel("number of sequences")
for i, v in enumerate(counts.values):
    axes[0].text(v + 0.3, i, str(v), va="center", fontsize=9)

order = list(counts.index)
unl = [counts[c] - labeled_counts.get(c, 0) for c in order]
lab = [labeled_counts.get(c, 0) for c in order]
axes[1].barh(order, unl, color="#cbd5e1", label="unlabeled (pretraining)")
axes[1].barh(order, lab, left=unl, color="#ef4444", label="labeled (the scarce 68)")
axes[1].set_title("Labeled vs unlabeled split")
axes[1].set_xlabel("number of sequences")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

n_labeled = int(manifest["is_labeled"].sum())
print(f"Labeled subset: {n_labeled} sequences (target 68 on the full dataset).")
print(f"Unlabeled pool: {len(manifest) - n_labeled} sequences for pretraining.")

## One video can hold several sequences

A single YouTube clip often shows more than one walking bout, or the same walk
labeled from two camera sides, so several sequences can point at the same video
id. That matters for the next notebook: to be polite to YouTube and to save disk
and time, we download each unique video only once and then reuse it for every
sequence that references it. The count below is why bulk downloading dedups by
video id rather than by sequence.

In [ ]:
per_video = manifest.groupby("video_id").size().sort_values(ascending=False)
shared = per_video[per_video > 1]
print(f"Unique videos referenced : {manifest['video_id'].nunique()}")
print(f"Total sequences          : {len(manifest)}")
print(f"Videos with >1 sequence  : {len(shared)}")
if len(shared):
    print("\nA few videos shared by multiple sequences:")
    for vid, n in shared.head(5).items():
        print(f"  {vid}: {n} sequences")

## Save the manifest for the rest of the series

We write the manifest to the cache folder as a CSV. Notebook 01 reads it to know
which videos to download, notebook 02 reads it to know which sequences to extract,
and notebook 03 reads the `is_labeled` column to hold out the 68-clip probe set.
Saving one small table here means every later notebook starts from the same map
of the dataset.

In [ ]:
ns = CONFIG["CACHE_NS"]
manifest_path = CONFIG["CACHE_DIR"] / f"manifest{ns}.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Wrote manifest to {manifest_path}")
print(f"  {len(manifest)} sequences, {manifest['video_id'].nunique()} unique videos, "
      f"{int(manifest['is_labeled'].sum())} labeled")
print("Columns:", list(manifest.columns))

# The 68-row labelled manifest (locked run) so downstream notebooks can audit and
# re-verify the exp5 lock from cache alone, plus a fingerprint of the locked id set.
labeled_manifest = manifest[manifest["is_labeled"]].copy()
labeled_manifest_path = CONFIG["CACHE_DIR"] / f"labeled_manifest{ns}.csv"
labeled_manifest.to_csv(labeled_manifest_path, index=False)

if LABELED_ID2CLASS is not None:
    CANONICAL_ID_HASH = canonical_id_hash(LABELED_ID2CLASS.keys())
    # Persist exp5's EXACT seed-42 47/21 split over exp5's list order so notebook 05 can
    # report a like-for-like accuracy point beside 0.76 on the same partition exp5 used.
    train_ids, test_ids = exp5_exact_split(EXP5_LIST_ORDER, train_portion=0.7)
    import pandas as _pd
    split_rows = ([{"seq": s, "fold": "train"} for s in train_ids]
                  + [{"seq": s, "fold": "test"} for s in test_ids])
    _pd.DataFrame(split_rows).to_csv(CONFIG["CACHE_DIR"] / f"exp5_split{ns}.csv", index=False)
    print(f"Wrote labeled_manifest{ns}.csv ({len(labeled_manifest)} rows) and "
          f"exp5_split{ns}.csv ({len(train_ids)} train / {len(test_ids)} test).")
else:
    CANONICAL_ID_HASH = "smoke-or-firstN"
    print(f"Wrote labeled_manifest{ns}.csv ({len(labeled_manifest)} rows). "
          f"No exp5 split persisted (not a locked run).")

(CONFIG["CACHE_DIR"] / f"canonical_id_hash{ns}.txt").write_text(CANONICAL_ID_HASH)
print(f"Canonical id hash: {CANONICAL_ID_HASH}  (stamped onto every cache artifact)")

## Recap and what comes next

We turned a sprawling folder of sequence CSVs into a single manifest table, one
row per sequence, and we saw the imbalance that motivates the whole series: a
large unlabeled pool and a tiny labeled slice. We also saw that videos are shared
across sequences, which is why the next step dedups before downloading.

In notebook 01 we take the unique video ids from this manifest and download each
one, resuming cleanly so already-cached videos are skipped. That fills the video
cache that notebook 02 will run MediaPipe over to produce the skeletons at the
heart of the JEPA.